# Repeated-seed training stability for internal GRF models

This notebook retrains each **available completed outer-fold configuration** from scratch with five seeds. Architecture, hyperparameters, training duration, preprocessing and outer test participant stay fixed. It does not rerun hyperparameter selection, choose a best seed, train on external data or overwrite original models.

**Important scope:** your previously completed nested analysis supports AP-left/SUB01 LSTM and GRU. If only those configurations are available, the result is a one-participant sensitivity analysis—not a complete cohort-wide repeat and not a repeat of all original manuscript models. Other tasks/folds/Conv1D-LSTM are included automatically only when their original selection and refit metadata are complete.

1. Upload this notebook to Colab and select a GPU runtime.
2. Use the same Google Drive account as the earlier run. Keep the original `Shida-LOSO-ANKLE.zip` in `MyDrive/Shida_Nested_LOSO/datasets/` and the completed experiment under `complete_studies/reviewer_complete_v1/`.
3. Run configuration and setup. If packages are changed, restart the runtime once, then run all cells again.
4. Inspect the inventory before running training. Missing settings are never invented.
5. Run training and then reports/export. Progress is saved to Drive **after every complete epoch**, including optimizer state; restarting resumes the last complete epoch.

The seed list is fixed before training. Never select or report only the best seed. Reports include failed/missing runs and use the same set of participants for each seed. This measures conditional training stability with frozen settings, not uncertainty from repeating hyperparameter search.

In [ ]:
from pathlib import Path
import json, re, sys, os
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/Shida_Nested_LOSO')
DATA_ZIP = PROJECT/'datasets/Shida-LOSO-ANKLE.zip'
SOURCE_ROOT = PROJECT/'complete_studies/reviewer_complete_v1'
RUN_ID = 'seed_stability_v1'
TASKS = ['AP_Left','AP_Right','Vertical_Left','Vertical_Right','ML_Left','ML_Right']
FAMILIES = ['lstm','gru','conv1d_lstm']
OUTER_PARTICIPANTS = 'all'     # Or ['SUB01']; never select folds by their test errors.
SEEDS = [11,22,33,44,55]
INSTALL_MATCHING_PACKAGES = True
TRAIN_VERBOSE = 0             # Progress printed at epoch 1 and every 10 epochs.
OUTPUT = PROJECT/'seed_stability'/RUN_ID
CODE = Path('/content/shida_seed_stability_code')
assert re.fullmatch(r'[A-Za-z0-9_-]+',RUN_ID)
assert len(SEEDS)>=3 and len(set(SEEDS))==len(SEEDS) and all(type(s)==int and 0<=s<2**31 for s in SEEDS)
if not DATA_ZIP.is_file():raise FileNotFoundError(f'Place the original paired-input ZIP at {DATA_ZIP}')
spec_paths = [SOURCE_ROOT/t/'absolute/prespecification.json' for t in TASKS]
spec_paths = [p for p in spec_paths if p.is_file()]
if not spec_paths:raise FileNotFoundError('No completed-source prespecification files found; correct SOURCE_ROOT. Internal.rar contains predictions, not retraining settings or ankle inputs.')
reference = json.loads(spec_paths[0].read_text())
print('Original results:',SOURCE_ROOT)
print('New checkpoint/output directory:',OUTPUT)
print('Requested master seeds:',SEEDS)


In [ ]:
# Match the software used for selecting the original configurations.
import importlib.metadata, subprocess
required = {}
for name in ['numpy','pandas','scikit-learn','tensorflow','keras','keras-tuner','joblib']:
    versions={json.loads(p.read_text())['software'].get(name) for p in spec_paths}
    if len(versions)!=1 or None in versions:raise ValueError(f'Mixed or unavailable source {name} versions; run compatible experiments separately.')
    required[name]=versions.pop()
changes=[]
for name,version in required.items():
    try: actual=importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:actual=None
    if actual!=version:changes.append(f'{name}=={version}')
if changes:
    if not INSTALL_MATCHING_PACKAGES:raise RuntimeError('Install matching source package versions: '+str(changes))
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',*changes,'openpyxl','matplotlib'])
    raise RuntimeError('Packages changed. Restart the Colab runtime, then Run all again. Drive results are retained.')
print('Source software versions matched:',required)


In [ ]:
# Write bundled source definitions; never modify the original experiment.
CODE.mkdir(parents=True,exist_ok=True)
BUNDLED_FILES = {'data.py': '"""Read the paired per-participant Raw workbooks in Shida-LOSO-ANKLE."""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom hashlib import sha256\nfrom io import BytesIO\nfrom pathlib import Path\nimport re\nfrom zipfile import ZipFile\n\nimport numpy as np\nimport pandas as pd\n\n\nTASKS = {\n    "apleft": "AP_Left", "apright": "AP_Right",\n    "longlef": "Vertical_Left", "longleft": "Vertical_Left",\n    "longrig": "Vertical_Right", "longright": "Vertical_Right",\n    "verticalleft": "Vertical_Left", "verticalright": "Vertical_Right",\n    "mlleft": "ML_Left", "mlright": "ML_Right",\n}\nTASK_ORDER = ["AP_Left", "AP_Right", "Vertical_Left", "Vertical_Right", "ML_Left", "ML_Right"]\nRAW_PATTERN = re.compile(r"(?:^|/)Raw/([^/]+)/([^/]+\\.xlsx)$", re.I)\nSUBJECT_PATTERN = re.compile(r"(?:SUB|UB)(\\d+)", re.I)\n\n\n@dataclass\nclass Dataset:\n    task: str\n    x: np.ndarray                       # [cycles, 100 time points, 1 feature]\n    y: np.ndarray\n    groups: np.ndarray                  # participant ID for EVERY cycle\n    cycles: pd.DataFrame                # source and cycle-position provenance\n    fingerprint: str\n\n    @property\n    def subjects(self):\n        return sorted(np.unique(self.groups).tolist())\n\n    def subset(self, subjects):\n        mask = np.isin(self.groups, subjects)\n        if not mask.any():\n            raise ValueError("Empty participant subset")\n        fingerprint = sha256((self.fingerprint + repr(sorted(subjects))).encode()).hexdigest()\n        return Dataset(self.task, self.x[mask], self.y[mask], self.groups[mask],\n                       self.cycles.loc[mask].reset_index(drop=True), fingerprint)\n\n\ndef canonical_subject(text):\n    matches = SUBJECT_PATTERN.findall(str(text))\n    ids = {f"SUB{int(value):02d}" for value in matches}\n    if len(ids) != 1:\n        raise ValueError(f"Expected one unambiguous participant in {text!r}; got {ids}")\n    return ids.pop()\n\n\ndef workbook_arrays(payload, filename, subject, expected_timesteps=100):\n    """Keep the original in/out column order. GRF headers lack trial identifiers."""\n    with pd.ExcelFile(BytesIO(payload), engine="openpyxl") as book:\n        if not {"in", "out"}.issubset(book.sheet_names):\n            raise ValueError(f"{filename}: paired \'in\' and \'out\' sheets are required")\n        inputs = pd.read_excel(book, sheet_name="in", header=0)\n        targets = pd.read_excel(book, sheet_name="out", header=0)\n    if inputs.shape != targets.shape or inputs.shape[0] != expected_timesteps or not inputs.shape[1]:\n        raise ValueError(f"{filename}: expected matched ({expected_timesteps}, cycles) sheets; "\n                         f"got in={inputs.shape}, out={targets.shape}")\n    for header in inputs.columns:\n        if canonical_subject(header) != subject:\n            raise ValueError(f"{filename}: input column belongs to a different participant: {header}")\n    try:\n        x = inputs.to_numpy(dtype=np.float32).T[:, :, None]\n        y = targets.to_numpy(dtype=np.float32).T[:, :, None]\n    except (ValueError, TypeError) as exc:\n        raise ValueError(f"{filename}: nonnumeric waveform value") from exc\n    if not np.isfinite(x).all() or not np.isfinite(y).all():\n        raise ValueError(f"{filename}: missing/nonfinite values; no silent zero filling or exclusion is allowed")\n    metadata = [{"subject": subject, "cycle_id": f"{subject}:C{i+1:03d}",\n                 "column_position_1based": i + 1, "input_header": str(inputs.columns[i]),\n                 "target_header": str(targets.columns[i]), "source_workbook": filename}\n                for i in range(len(x))]\n    return x, y, metadata\n\n\ndef load_datasets(source, requested_tasks=None):\n    """Accept the uploaded ZIP, its extracted root, or the Raw directory itself.\n\n    Never load the preassembled Train files: they repeat subjects across folds\n    and some supplied input matrices were concatenated along the wrong axis.\n    """\n    source = Path(source).expanduser().resolve()\n    requested = set(requested_tasks or TASK_ORDER)\n    unknown = requested - set(TASK_ORDER)\n    if unknown:\n        raise ValueError(f"Unknown tasks: {sorted(unknown)}")\n    archive = None\n    if source.is_file() and source.suffix.lower() == ".zip":\n        archive = ZipFile(source)\n        names = sorted(n for n in archive.namelist() if RAW_PATTERN.search(n))\n        reader = archive.read\n    elif source.is_dir():\n        paths = {p.as_posix(): p for p in source.rglob("*.xlsx") if RAW_PATTERN.search(p.as_posix())}\n        names = sorted(paths)\n        reader = lambda name: paths[name].read_bytes()\n    else:\n        raise ValueError(f"Expected an existing ZIP or directory: {source}")\n    accum, seen, fingerprints = {}, set(), {}\n    try:\n        for name in names:\n            match = RAW_PATTERN.search(name)\n            task = TASKS.get(re.sub(r"[^a-z]", "", match.group(1).lower()))\n            if task not in requested:\n                continue\n            subject = canonical_subject(match.group(2))\n            if (task, subject) in seen:\n                raise ValueError(f"Duplicate Raw workbook for {task}/{subject}; provide one dataset root")\n            seen.add((task, subject))\n            payload = reader(name)\n            relative_name = "Raw/" + name.split("/")[-2] + "/" + name.split("/")[-1]\n            x, y, metadata = workbook_arrays(payload, relative_name, subject)\n            accum.setdefault(task, []).append((subject, x, y, metadata))\n            fingerprints.setdefault(task, sha256()).update(relative_name.encode() + sha256(payload).digest())\n    finally:\n        if archive is not None:\n            archive.close()\n    absent = requested - set(accum)\n    if absent:\n        raise ValueError(f"No Raw workbooks found for {sorted(absent)} in {source}")\n    result = {}\n    for task in TASK_ORDER:\n        if task not in accum:\n            continue\n        records = sorted(accum[task], key=lambda item: item[0])\n        if len(records) < 3:\n            raise ValueError(f"{task}: at least three participants are required for nested LOSO")\n        result[task] = Dataset(\n            task, np.concatenate([r[1] for r in records]), np.concatenate([r[2] for r in records]),\n            np.concatenate([np.repeat(r[0], len(r[1])) for r in records]),\n            pd.DataFrame([m for r in records for m in r[3]]), fingerprints[task].hexdigest())\n    return result\n\n\ndef nested_indices(groups, outer_subject):\n    """Yield train, validation, test indices; one entire subject in each holdout."""\n    groups = np.asarray(groups)\n    subjects = sorted(np.unique(groups).tolist())\n    if outer_subject not in subjects or len(subjects) < 3:\n        raise ValueError("Unknown outer participant or fewer than three participants")\n    test = np.flatnonzero(groups == outer_subject)\n    for validation_subject in subjects:\n        if validation_subject == outer_subject:\n            continue\n        validation = np.flatnonzero(groups == validation_subject)\n        train = np.flatnonzero((groups != outer_subject) & (groups != validation_subject))\n        assert_subject_separation(groups, train, validation, test)\n        yield validation_subject, train, validation, test\n\n\ndef assert_subject_separation(groups, train, validation, test):\n    combined = np.concatenate([train, validation, test])\n    if not np.array_equal(np.sort(combined), np.arange(len(groups))):\n        raise ValueError("Each cycle must appear exactly once across the three partitions")\n    sets = [set(np.asarray(groups)[idx]) for idx in [train, validation, test]]\n    if any(not s for s in sets) or sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2]:\n        raise ValueError("Participant overlap or an empty train/validation/test partition")\n\n\ndef audit_datasets(datasets):\n    tasks, participants, cycles, splits = [], [], [], []\n    for task, ds in datasets.items():\n        tasks.append({"task": task, "n_participants": len(ds.subjects), "n_cycles": len(ds.x),\n                      "timepoints": ds.x.shape[1], "features": ds.x.shape[2],\n                      "missing_input_values": int((~np.isfinite(ds.x)).sum()),\n                      "missing_target_values": int((~np.isfinite(ds.y)).sum()),\n                      "subjects": ds.subjects, "fingerprint": ds.fingerprint})\n        cycles.extend(ds.cycles.assign(task=task).to_dict("records"))\n        for subject in ds.subjects:\n            participants.append({"task": task, "subject": subject,\n                                 "n_cycles": int(np.sum(ds.groups == subject))})\n            for val_subject, tr, va, te in nested_indices(ds.groups, subject):\n                splits.append({"task": task, "outer_test_subject": subject,\n                               "inner_validation_subject": val_subject,\n                               "inner_training_subjects": sorted(set(ds.groups[tr])),\n                               "n_train_cycles": len(tr), "n_validation_cycles": len(va),\n                               "n_test_cycles": len(te)})\n    return {"tasks": tasks, "participants": participants, "cycles": cycles, "splits": splits}\n', 'protocol.py': '"""Backend-independent nested LOSO protocol, metrics, and resumable records."""\nfrom __future__ import annotations\n\nfrom hashlib import sha256\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nfrom data import Dataset, nested_indices\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef stable_seed(seed, *labels):\n    text = "|".join(map(str, (seed, *labels)))\n    return int.from_bytes(sha256(text.encode()).digest()[:4], "big") % (2**31 - 1)\n\n\ndef parameter_hash(parameters):\n    return sha256(json.dumps(parameters, sort_keys=True).encode()).hexdigest()\n\n\ndef scale_from_training(train, *others, enabled=False):\n    """Only training inputs determine scale. Targets retain their original units."""\n    if enabled:\n        mean = np.mean(train, axis=(0, 1), keepdims=True, dtype=np.float64)\n        std = np.std(train, axis=(0, 1), keepdims=True, dtype=np.float64)\n        std = np.where(std < 1e-8, 1.0, std)\n    else:\n        mean = np.zeros((1, 1, train.shape[-1]))\n        std = np.ones_like(mean)\n    arrays = [((arr - mean) / std).astype(np.float32) for arr in (train,) + others]\n    return arrays, {"enabled": enabled, "mean": mean.reshape(-1).tolist(),\n                    "std": std.reshape(-1).tolist()}\n\n\ndef cycle_metrics(truth, prediction):\n    if truth.shape != prediction.shape or not np.isfinite(prediction).all():\n        raise ValueError("Invalid prediction shape or nonfinite predictions")\n    error = truth.astype(np.float64) - prediction.astype(np.float64)\n    rmse = np.sqrt(np.mean(error**2, axis=(1, 2)))\n    mae = np.mean(np.abs(error), axis=(1, 2))\n    span = np.ptp(truth.astype(np.float64), axis=1).reshape(-1)\n    nrmse = np.divide(rmse, span, out=np.full_like(rmse, np.nan), where=span > 1e-12)\n    return pd.DataFrame({"rmse": rmse, "nrmse": nrmse, "mae": mae,\n                         "target_range": span, "nrmse_defined": span > 1e-12})\n\n\ndef evaluate_candidate(ds, outer_subject, parameters, backend, seed, directory, scale_input=False):\n    """Score one candidate equally across inner validation participants.\n\n    The backend only receives inner train/validation data. Outer test arrays\n    are never supplied to tuning, early stopping, or input scaling.\n    """\n    directory = Path(directory)\n    signature = parameter_hash(parameters)\n    records = []\n    for validation_subject, tr, va, _ in nested_indices(ds.groups, outer_subject):\n        path = directory / f"inner_{validation_subject}.json"\n        if path.exists():\n            record = read_json(path)\n            if record["parameter_hash"] != signature:\n                raise ValueError(f"Cached trial has different parameters: {path}")\n            records.append(record)\n            continue\n        (xtr, xva), scaler = scale_from_training(ds.x[tr], ds.x[va], enabled=scale_input)\n        fit_seed = stable_seed(seed, ds.task, outer_subject, validation_subject, "inner")\n        fit = backend.fit_inner(xtr, ds.y[tr], xva, ds.y[va], parameters, fit_seed,\n                                directory / f"history_{validation_subject}.csv")\n        metric = cycle_metrics(ds.y[va], fit["prediction"])\n        record = {"outer_subject": outer_subject, "validation_subject": validation_subject,\n                  "training_subjects": sorted(set(ds.groups[tr])), "n_training_cycles": len(tr),\n                  "n_validation_cycles": len(va), "seed": fit_seed,\n                  "parameter_hash": signature, "scaler": scaler,\n                  "best_epoch": int(fit["best_epoch"]), "epochs_run": int(fit["epochs_run"]),\n                  "validation_mae": float(metric.mae.mean()),\n                  "validation_rmse": float(metric.rmse.mean())}\n        write_json(path, record)\n        records.append(record)\n    score = float(np.mean([r["validation_mae"] for r in records]))\n    if not np.isfinite(score):\n        raise ValueError("Nonfinite inner objective")\n    summary = {"parameters": parameters, "inner_subject_mae": score, "folds": records}\n    write_json(directory / "inner_summary.json", summary)\n    return summary\n\n\ndef refit_and_evaluate(ds, outer_subject, parameters, inner_summary, backend, seed,\n                       directory, architecture, scale_input=False):\n    """Refit on ALL outer-training participants for an inner-derived epoch budget."""\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    tr = np.flatnonzero(ds.groups != outer_subject)\n    te = np.flatnonzero(ds.groups == outer_subject)\n    if len(te) == 0 or set(ds.groups[tr]) & set(ds.groups[te]):\n        raise ValueError("Invalid outer subject separation")\n    if set(r["validation_subject"] for r in inner_summary["folds"]) != set(ds.groups[tr]):\n        raise ValueError("The winning trial did not validate every outer-training participant")\n    if any(r["outer_subject"] != outer_subject for r in inner_summary["folds"]):\n        raise ValueError("Inner summary belongs to a different outer fold")\n    if parameter_hash(inner_summary["parameters"]) != parameter_hash(parameters):\n        raise ValueError("Refit parameters differ from the selected inner trial")\n    # Predefined rule: ceiling of median inner best epoch; outer test never selects epochs.\n    epochs = max(1, int(np.ceil(np.median([r["best_epoch"] for r in inner_summary["folds"]]))))\n    (xtr, xte), scaler = scale_from_training(ds.x[tr], ds.x[te], enabled=scale_input)\n    refit_seed = stable_seed(seed, ds.task, outer_subject, "refit")\n    model = backend.fit_final(xtr, ds.y[tr], parameters, refit_seed, epochs,\n                              directory / "final_history.csv")\n    # This is the first point where the fitted outer model sees test inputs.\n    prediction = backend.predict(model, xte)\n    metrics = cycle_metrics(ds.y[te], prediction)\n    metadata = ds.cycles.iloc[te].reset_index(drop=True)\n    per_cycle = pd.concat([metadata, metrics], axis=1)\n    per_cycle.insert(0, "task", ds.task)\n    per_cycle.insert(1, "architecture", architecture)\n    per_cycle.insert(2, "seed", seed)\n    per_cycle.to_csv(directory / "cycle_metrics.csv", index=False)\n    np.savez_compressed(directory / "predictions.npz", measured=ds.y[te], predicted=prediction,\n                        cycle_ids=metadata.cycle_id.to_numpy(dtype=str),\n                        subject_ids=ds.groups[te], gait_phase=np.linspace(0, 100, ds.y.shape[1]))\n    backend.save(model, directory / "model.keras")\n    backend.clear()\n    summary = {"task": ds.task, "architecture": architecture, "seed": seed,\n               "subject": outer_subject, "n_cycles": len(te), "n_train_cycles": len(tr),\n               "n_train_subjects": len(set(ds.groups[tr])), "epochs": epochs,\n               "refit_seed": refit_seed, "rmse": float(metrics.rmse.mean()),\n               "nrmse": float(metrics.nrmse.mean()) if metrics.nrmse.notna().any() else None,\n               "mae": float(metrics.mae.mean()),\n               "undefined_nrmse_cycles": int((~metrics.nrmse_defined).sum()),\n               "inner_subject_mae": inner_summary["inner_subject_mae"],\n               "parameters": parameters, "input_scaler": scaler,\n               "epoch_rule": "ceiling of median best epoch across winning inner LOSO fits"}\n    # Written last: a completed outer fold can be safely skipped on resume.\n    write_json(directory / "result.json", summary)\n    return summary\n\n\ndef summarize_subject_results(records, directory, bootstrap_seed=2026, resamples=2000):\n    """Bootstrap participants; repeated seeds are averaged within participant first."""\n    directory = Path(directory)\n    df = pd.DataFrame([{k: v for k, v in r.items() if not isinstance(v, (list, dict))}\n                       for r in records])\n    if df.empty:\n        return\n    df.to_csv(directory / "outer_subject_metrics_by_seed.csv", index=False)\n    averaged = df.groupby(["task", "architecture", "subject"], as_index=False).agg(\n        rmse=("rmse", "mean"), nrmse=("nrmse", "mean"), mae=("mae", "mean"),\n        n_seeds=("seed", "nunique"), rmse_seed_sd=("rmse", "std"),\n        nrmse_seed_sd=("nrmse", "std"), mae_seed_sd=("mae", "std"))\n    averaged.to_csv(directory / "outer_subject_metrics_seed_average.csv", index=False)\n    rows = []\n    for (task, arch), group in averaged.groupby(["task", "architecture"]):\n        for metric in ["rmse", "nrmse", "mae"]:\n            values = group[metric].dropna().to_numpy()\n            if not len(values):\n                continue\n            rng = np.random.default_rng(stable_seed(bootstrap_seed, task, arch, metric))\n            if len(values) >= 2:\n                boot = np.median(rng.choice(values, size=(resamples, len(values)), replace=True), axis=1)\n                low, high = np.quantile(boot, [0.025, 0.975])\n            else:\n                low = high = np.nan\n            rows.append({"task": task, "architecture": arch, "metric": metric,\n                         "n_participants": len(values), "min_seeds_per_subject": int(group.n_seeds.min()),\n                         "max_seeds_per_subject": int(group.n_seeds.max()),\n                         "median": float(np.median(values)), "q25": float(np.quantile(values, .25)),\n                         "q75": float(np.quantile(values, .75)),\n                         "median_ci95_lower": low, "median_ci95_upper": high,\n                         "ci_method": "participant percentile bootstrap after averaging seeds"})\n    pd.DataFrame(rows).to_csv(directory / "cohort_summary.csv", index=False)\n', 'models.py': '"""TensorFlow model definitions and Bayesian selection over complete inner LOSO.\n\nTensorFlow/KerasTuner are imported only during training. Audit and data tests\ntherefore work in a lightweight environment without a GPU or TensorFlow.\n"""\nfrom __future__ import annotations\n\nimport gc\nfrom pathlib import Path\nimport numpy as np\n\nfrom protocol import evaluate_candidate, read_json, stable_seed, write_json\n\n\ndef training_dependencies():\n    try:\n        import tensorflow as tf\n        import keras_tuner as kt\n    except ImportError as exc:\n        raise RuntimeError("Training requires TensorFlow and keras-tuner. Install requirements.txt "\n                           "in a Python 3.11 environment; audit mode needs only requirements-audit.txt.") from exc\n    return tf, kt\n\n\ndef configure_tensorflow(seed, threads=0):\n    tf, _ = training_dependencies()\n    if threads:\n        tf.config.threading.set_inter_op_parallelism_threads(threads)\n        tf.config.threading.set_intra_op_parallelism_threads(threads)\n    for device in tf.config.list_physical_devices("GPU"):\n        tf.config.experimental.set_memory_growth(device, True)\n    tf.keras.utils.set_random_seed(seed)\n    tf.config.experimental.enable_op_determinism()\n    return {"tensorflow": tf.__version__, "devices": [d.name for d in tf.config.list_physical_devices()]}\n\n\ndef search_space(architecture, profile):\n    _, kt = training_dependencies()\n    hp = kt.HyperParameters()\n    if profile == "smoke":\n        hp.Fixed("num_layers", 1)\n        hp.Fixed("activation", "tanh")\n        hp.Fixed("optimizer", "adam")\n        hp.Fixed("learning_rate", 0.001)\n        hp.Fixed("loss", "mae")\n        hp.Fixed("encoder_units", 4)\n        hp.Fixed("decoder_units", 4)\n        hp.Fixed("units_0", 4)\n        return hp\n    max_layers = 10 if profile == "original" else 3\n    hp.Int("num_layers", 1, max_layers)\n    # The original GRU code registers tanh first under the shared HP name.\n    hp.Choice("activation", ["relu", "tanh"] if architecture == "conv1d_lstm" and profile == "original" else ["tanh"])\n    hp.Choice("optimizer", ["adam", "sgd", "rmsprop"] if profile == "original" else ["adam"])\n    hp.Choice("learning_rate", [0.0001, 0.001, 0.01] if profile == "original" else [0.0001, 0.001])\n    hp.Choice("loss", ["mse", "mae", "huber"] if architecture == "gru" else ["mse", "mae"])\n    hp.Fixed("encoder_units", (32 if architecture == "gru" else 100) if profile == "original" else 32)\n    if architecture != "gru":\n        hp.Choice("decoder_units", list(range(32, 513, 32)) if profile == "original" else [32, 64, 128])\n    unit_options = (list(range(32, 513, 32)) if architecture == "gru" else list(range(100, 513, 32)))\n    if profile != "original":\n        unit_options = [32, 64, 128]\n    for layer in range(max_layers):\n        with hp.conditional_scope("num_layers", list(range(layer + 1, max_layers + 1))):\n            hp.Choice(f"units_{layer}", unit_options)\n    return hp\n\n\ndef build_model(architecture, timesteps, parameters, profile="original"):\n    tf, _ = training_dependencies()\n    layers = tf.keras.layers\n    model = tf.keras.Sequential(name=architecture)\n    model.add(layers.Input(shape=(timesteps, 1)))\n    activation = parameters["activation"]\n    if architecture == "conv1d_lstm":\n        model.add(layers.Conv1D(4 if profile == "smoke" else 64, 3, activation="relu"))\n        model.add(layers.MaxPooling1D(pool_size=2))\n    if architecture in {"lstm", "conv1d_lstm"}:\n        model.add(layers.LSTM(parameters["encoder_units"], activation=activation))\n        model.add(layers.RepeatVector(timesteps))\n        model.add(layers.LSTM(parameters["decoder_units"], activation=activation, return_sequences=True))\n        for layer in range(parameters["num_layers"]):\n            model.add(layers.LSTM(parameters[f"units_{layer}"], activation=activation, return_sequences=True))\n        model.add(layers.TimeDistributed(layers.Dense(1)))\n    elif architecture == "gru":\n        model.add(layers.GRU(parameters["encoder_units"], activation=activation, return_sequences=True))\n        for layer in range(parameters["num_layers"]):\n            model.add(layers.GRU(parameters[f"units_{layer}"], activation=activation, return_sequences=True))\n        model.add(layers.Dense(1))\n    else:\n        raise ValueError(f"Unknown architecture: {architecture}")\n    optimizer = {"adam": tf.keras.optimizers.Adam, "sgd": tf.keras.optimizers.SGD,\n                 "rmsprop": tf.keras.optimizers.RMSprop}[parameters["optimizer"]](\n                     learning_rate=parameters["learning_rate"])\n    loss = tf.keras.losses.Huber() if parameters["loss"] == "huber" else parameters["loss"]\n    model.compile(optimizer=optimizer, loss=loss, metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")])\n    return model\n\n\nclass TensorFlowBackend:\n    def __init__(self, architecture, timesteps, profile, epochs, patience, batch_size, verbose):\n        self.tf, _ = training_dependencies()\n        self.architecture, self.timesteps, self.profile = architecture, timesteps, profile\n        self.epochs, self.patience = epochs, patience\n        self.batch_size, self.verbose = batch_size, verbose\n\n    def clear(self):\n        self.tf.keras.backend.clear_session()\n        gc.collect()\n\n    def fresh(self, parameters, seed):\n        self.clear()\n        self.tf.keras.utils.set_random_seed(seed)\n        return build_model(self.architecture, self.timesteps, parameters, self.profile)\n\n    def fit_inner(self, x_train, y_train, x_validation, y_validation, parameters, seed, history_path):\n        history_path = Path(history_path)\n        history_path.parent.mkdir(parents=True, exist_ok=True)\n        model = self.fresh(parameters, seed)\n        tf = self.tf\n        stop = tf.keras.callbacks.EarlyStopping(monitor="val_mae", mode="min",\n                                               patience=self.patience, min_delta=0,\n                                               restore_best_weights=True)\n        try:\n            history = model.fit(x_train, y_train,\n                                validation_data=(x_validation, y_validation),\n                                epochs=self.epochs, batch_size=self.batch_size, shuffle=True,\n                                callbacks=[stop, tf.keras.callbacks.TerminateOnNaN(),\n                                           tf.keras.callbacks.CSVLogger(str(history_path))],\n                                verbose=self.verbose)\n            values = np.asarray(history.history["val_mae"], dtype=float)\n            if not len(values) or not np.isfinite(values).all():\n                raise ValueError("Nonfinite inner training/validation history; trial must fail")\n            best_epoch = int(np.argmin(values)) + 1\n            prediction = self.predict(model, x_validation)\n            return {"prediction": prediction, "best_epoch": best_epoch, "epochs_run": len(values)}\n        finally:\n            del model\n            self.clear()\n\n    def fit_final(self, x_train, y_train, parameters, seed, epochs, history_path):\n        """No validation split or validation-driven callback in the outer refit."""\n        model = self.fresh(parameters, seed)\n        tf = self.tf\n        history = model.fit(x_train, y_train, epochs=epochs, batch_size=self.batch_size,\n                            shuffle=True, callbacks=[tf.keras.callbacks.TerminateOnNaN(),\n                                                      tf.keras.callbacks.CSVLogger(str(history_path))],\n                            verbose=self.verbose)\n        if len(history.history["loss"]) != epochs or not np.isfinite(history.history["loss"]).all():\n            raise ValueError("Outer refit did not complete with finite losses")\n        return model\n\n    def predict(self, model, inputs):\n        return np.asarray(model.predict(inputs, batch_size=self.batch_size, verbose=0))\n\n    def save(self, model, path):\n        model.save(path)\n        lines = []\n        model.summary(print_fn=lambda line, **kwargs: lines.append(line))\n        Path(path).with_suffix(".summary.txt").write_text("\\n".join(lines), encoding="utf-8")\n\n\ndef select_hyperparameters(ds, outer_subject, architecture, backend, seed, directory,\n                           profile, max_trials, scale_input, resume=False):\n    """A Bayesian trial is a COMPLETE inner LOSO, never just one validation subject."""\n    _, kt = training_dependencies()\n    directory = Path(directory)\n    trial_root = directory / "candidate_fits"\n    trial_root.mkdir(parents=True, exist_ok=True)\n\n    class InnerLOSOTuner(kt.Tuner):\n        def run_trial(self, trial, *args, **kwargs):\n            parameters = dict(trial.hyperparameters.values)\n            trial_dir = trial_root / trial.trial_id\n            print(f"  {ds.task}/{architecture}/{outer_subject}: trial {trial.trial_id}", flush=True)\n            try:\n                summary = evaluate_candidate(ds, outer_subject, parameters, backend, seed,\n                                             trial_dir, scale_input)\n            except (ValueError, backend.tf.errors.ResourceExhaustedError) as exc:\n                write_json(trial_dir / "failure.json", {"error": str(exc), "parameters": parameters})\n                # Do not average only the successful validation participants.\n                raise kt.errors.FailedTrialError(str(exc)) from exc\n            return {"inner_subject_mae": summary["inner_subject_mae"]}\n\n    hp = search_space(architecture, profile)\n    oracle = kt.oracles.BayesianOptimizationOracle(\n        objective=kt.Objective("inner_subject_mae", direction="min"),\n        max_trials=max_trials, num_initial_points=min(3, max_trials),\n        hyperparameters=hp, allow_new_entries=False, tune_new_entries=False,\n        seed=stable_seed(seed, ds.task, architecture, outer_subject, "oracle"),\n        max_retries_per_trial=0, max_consecutive_failed_trials=3)\n    tuner = InnerLOSOTuner(oracle=oracle, hypermodel=None, directory=str(directory),\n                          project_name="tuner", overwrite=not resume)\n    tuner.search()\n    best = tuner.oracle.get_best_trials(num_trials=1)\n    if not best or best[0].score is None or not np.isfinite(best[0].score):\n        raise RuntimeError("No successful full inner LOSO trial; inspect failure.json files")\n    trial = best[0]\n    summary = read_json(trial_root / trial.trial_id / "inner_summary.json")\n    write_json(directory / "selection.json", {\n        "trial_id": trial.trial_id, "inner_subject_mae": summary["inner_subject_mae"],\n        "parameters": summary["parameters"], "search_profile": profile,\n        "n_inner_folds": len(summary["folds"])})\n    return summary["parameters"], summary\n', 'revision_data.py': '"""Strict cohort contract and coordinate controls; no guessed anatomical data."""\nfrom dataclasses import dataclass, replace\nfrom hashlib import sha256\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nfrom data import Dataset, load_datasets\nfrom protocol import write_json, read_json\n\nMODES = (\'absolute\', \'translation\', \'pelvis\', \'aligned\', \'body\', \'combined\')\nAXIS = {\'AP\': 0, \'ML\': 1, \'Vertical\': 2}\nREQUIRED_CONTEXT = [\'cohort_id\', \'dataset_role\', \'ankle_units\', \'grf_units\',\n                    \'coordinate_definition\', \'motion_capture_system\', \'force_platforms\',\n                    \'kinematics_sampling_hz\', \'force_sampling_hz\', \'walking_protocol\',\n                    \'harmonization_description\', \'cycle_definition\']\n\ndef digest_file(path):\n    h = sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\ndef bool_column(series):\n    mapping = {\'true\': True, \'1\': True, \'yes\': True, \'false\': False, \'0\': False, \'no\': False}\n    v = series.astype(str).str.lower().str.strip().map(mapping)\n    if v.isna().any():\n        raise ValueError(\'include must contain explicit true/false values\')\n    return v.astype(bool)\n\ndef read_table(path, columns):\n    df = pd.read_csv(path, dtype=str, keep_default_na=False)\n    if set(columns) - set(df):\n        raise ValueError(f\'{path}: missing columns {set(columns) - set(df)}\')\n    for col in set(columns) - {\'exclusion_reason\'}:\n        if df[col].str.strip().eq(\'\').any():\n            raise ValueError(f\'{path}: blank required {col}\')\n    return df\n\n@dataclass\nclass Cohort:\n    ds: Dataset\n    context: dict\n    participants: pd.DataFrame\n    trials: pd.DataFrame\n    ledger: pd.DataFrame\n    arrays: dict\n\ndef legacy_cohort(archive, task):\n    ds = load_datasets(archive, [task])[task]\n    meta = ds.cycles.copy()\n    meta[\'participant_id\'] = meta.subject\n    meta[\'trial_id\'] = \'unknown\'  # Never invent trial IDs from cycle positions.\n    meta[\'visit\'] = \'internal\'\n    meta[\'side\'] = task.split(\'_\')[-1].lower()\n    meta[\'include\'] = True\n    meta[\'exclusion_reason\'] = \'\'\n    ds = replace(ds, cycles=meta)\n    context = {k: \'unknown\' for k in REQUIRED_CONTEXT}\n    context.update(cohort_id=\'Shida_internal\', dataset_role=\'internal\', ankle_units=\'mm\',\n                   grf_units=\'N/kg\', units_verified=False,\n                   coordinate_definition=\'Original single component; physical frame unverified\',\n                   source=\'Paired Raw sheets; manuscript units assumed, not independently verified\')\n    people = pd.DataFrame({\'participant_id\': ds.subjects})\n    return Cohort(ds, context, people, pd.DataFrame(), meta, {})\n\ndef load_cohort(directory, task, visits=None, role=None):\n    """Read one side/component cohort bundle. Excluded cycles need no waveform."""\n    root = Path(directory)\n    context = read_json(root / \'cohort.json\')\n    if any(k not in context or context[k] is None for k in REQUIRED_CONTEXT):\n        raise ValueError(\'Complete cohort.json; use "unknown" explicitly for unavailable descriptive fields\')\n    if role and context[\'dataset_role\'] != role:\n        raise ValueError(\'Wrong dataset_role\')\n    if context[\'ankle_units\'] not in (\'m\', \'mm\') or context[\'grf_units\'] != \'N/kg\':\n        raise ValueError(\'Convert ankle units to m/mm and GRF to N/kg before making the bundle\')\n    if context.get(\'task\') != task:\n        raise ValueError(\'cohort.json task differs from requested component/side\')\n    people = read_table(root / \'participants.csv\', [\'participant_id\'])\n    trials = read_table(root / \'trials.csv\', [\'participant_id\', \'trial_id\', \'visit\', \'side\', \'include\', \'exclusion_reason\'])\n    ledger = read_table(root / \'cycles.csv\', [\'cycle_id\', \'participant_id\', \'trial_id\', \'visit\', \'side\', \'include\', \'exclusion_reason\'])\n    if people.participant_id.duplicated().any() or ledger.cycle_id.duplicated().any():\n        raise ValueError(\'Participant and cycle IDs must be unique within this cohort bundle\')\n    if not people.participant_id.str.fullmatch(r\'[A-Za-z0-9_-]+\').all():\n        raise ValueError(\'Use letters, digits, underscores, or hyphens in participant IDs\')\n    keys = [\'participant_id\', \'trial_id\', \'visit\', \'side\']\n    if trials.duplicated(keys).any():\n        raise ValueError(\'Duplicate trial record\')\n    for frame in (trials, ledger):\n        frame[\'include\'] = bool_column(frame.include)\n        if frame.loc[~frame.include, \'exclusion_reason\'].str.strip().eq(\'\').any():\n            raise ValueError(\'Every excluded trial/cycle requires a reason\')\n        if set(frame.participant_id) - set(people.participant_id):\n            raise ValueError(\'Participant missing from participants.csv\')\n        if not set(frame.visit).issubset({\'baseline\', \'post_intervention\', \'internal\'}):\n            raise ValueError(\'Use baseline, post_intervention, or internal visit labels\')\n        if set(frame.side) != {task.split(\'_\')[-1].lower()}:\n            raise ValueError(\'Each bundle must describe just its selected side\')\n    joined = ledger.merge(trials[keys + [\'include\']], on=keys, how=\'left\', suffixes=(\'\', \'_trial\'), validate=\'many_to_one\')\n    if joined.include_trial.isna().any() or (joined.include & ~joined.include_trial.astype(bool)).any():\n        raise ValueError(\'Every cycle must link to a trial; an excluded trial cannot have an included cycle\')\n    with np.load(root / \'waveforms.npz\', allow_pickle=False) as f:\n        arrays = {k: f[k] for k in f.files}\n    ids = arrays.pop(\'cycle_ids\').astype(str)\n    valid = ledger[ledger.include].copy().set_index(\'cycle_id\', drop=False)\n    if len(set(ids)) != len(ids) or set(ids) != set(valid.index):\n        raise ValueError(\'NPZ cycle_ids must match ALL included cycles, one-to-one, before visit filtering\')\n    meta = valid.loc[ids].reset_index(drop=True)\n    n = len(ids)\n    for k, arr in arrays.items():\n        if len(arr) != n or not np.issubdtype(arr.dtype, np.number) or not np.isfinite(arr).all():\n            raise ValueError(f\'{k}: wrong cycle count or nonfinite/nonnumeric values\')\n    if \'ankle_xyz\' in arrays:\n        xyz = arrays[\'ankle_xyz\']\n        if xyz.shape != (n, 100, 3) or arrays.get(\'grf_xyz\', np.empty(0)).shape != xyz.shape:\n            raise ValueError(\'ankle_xyz and grf_xyz must both be [N,100,3]\')\n        if context.get(\'canonical_axis_order\') != [\'forward\', \'left\', \'up\']:\n            raise ValueError(\'Map both kinematics and GRFs into the same right-handed [forward,left,up] laboratory axes\')\n        if arrays.get(\'direction_xyz\', np.empty(0)).shape != (n, 3):\n            raise ValueError(\'Provide a force-independent horizontal walking direction for every cycle\')\n        basis = gait_basis(arrays[\'direction_xyz\'])\n        axis = AXIS[task.split(\'_\')[0]]\n        x = xyz[:, :, axis:axis + 1]\n        # SAME gait-frame targets for ALL coordinate variants, enabling fair comparisons.\n        y = np.einsum(\'nti,nij->ntj\', arrays[\'grf_xyz\'], basis)[:, :, axis:axis + 1]\n    else:\n        x, y = arrays.get(\'x\'), arrays.get(\'y\')\n        if x is None or y is None or x.shape != (n, 100, 1) or y.shape != x.shape:\n            raise ValueError(\'Scalar bundles require x and y, each [N,100,1]\')\n    take = np.ones(n, bool) if visits is None else meta.visit.isin(visits).to_numpy()\n    if not take.any():\n        raise ValueError(\'No valid cycles for the prespecified visit policy\')\n    meta = meta.loc[take].reset_index(drop=True)\n    meta[\'subject\'] = meta.participant_id\n    fingerprint = sha256(\'\'.join(digest_file(root / f) for f in\n                         [\'cohort.json\', \'participants.csv\', \'trials.csv\', \'cycles.csv\', \'waveforms.npz\']).encode()).hexdigest()\n    fingerprint = sha256((fingerprint + repr(visits)).encode()).hexdigest()\n    ds = Dataset(task, x[take].astype(\'float32\'), y[take].astype(\'float32\'),\n                 meta.participant_id.to_numpy(str), meta, fingerprint)\n    return Cohort(ds, context, people, trials, ledger, {k: v[take] for k, v in arrays.items()})\n\ndef gait_basis(direction):\n    forward = np.array(direction, float, copy=True)\n    if forward.ndim != 2 or forward.shape[1] != 3 or not np.isfinite(forward).all():\n        raise ValueError(\'Invalid direction vectors\')\n    if np.any(np.abs(forward[:, 2]) > 1e-6):\n        raise ValueError(\'Walking directions must be projected onto the horizontal plane\')\n    length = np.linalg.norm(forward, axis=1)\n    if np.any(length < 1e-8):\n        raise ValueError(\'Zero walking direction; supply verified direction from pelvis progression/protocol\')\n    forward /= length[:, None]\n    up = np.tile([0., 0., 1.], (len(forward), 1))\n    left = np.cross(up, forward)\n    return np.stack([forward, left, up], axis=2)\n\ndef coordinate_dataset(cohort, mode=\'absolute\', translation_m=None, rotation_deg=0):\n    if mode not in MODES:\n        raise ValueError(f\'Unknown coordinate mode {mode}\')\n    ds, a = cohort.ds, cohort.arrays\n    units = cohort.context[\'ankle_units\']\n    factor = 1000. if units == \'mm\' else 1.\n    axis = AXIS[ds.task.split(\'_\')[0]]\n    if \'ankle_xyz\' not in a:\n        if mode not in (\'absolute\', \'translation\') or rotation_deg:\n            raise ValueError(\'This analysis requires verified 3D/pelvis/body-scale data; scalar inputs cannot supply it\')\n        x = ds.x.astype(float).copy()\n        if translation_m is not None:\n            if not cohort.context.get(\'units_verified\', False):\n                raise ValueError(\'Verify physical units before metre-based perturbations\')\n            x += np.asarray(translation_m)[axis] * factor\n        if mode == \'translation\':\n            x -= x[:, :1, :]\n    else:\n        x = a[\'ankle_xyz\'].astype(float).copy()\n        pelvis = a.get(\'pelvis_xyz\')\n        pelvis = None if pelvis is None else pelvis.astype(float).copy()\n        direction = a[\'direction_xyz\'].astype(float).copy()\n        if rotation_deg:\n            theta = np.deg2rad(rotation_deg)\n            R = np.array([[np.cos(theta), -np.sin(theta), 0], [np.sin(theta), np.cos(theta), 0], [0, 0, 1]])\n            x = x @ R.T\n            direction = direction @ R.T\n            if pelvis is not None:\n                pelvis = pelvis @ R.T\n        if translation_m is not None:\n            if not cohort.context.get(\'units_verified\', False):\n                raise ValueError(\'Verify physical units before metre-based perturbations\')\n            offset = np.asarray(translation_m, float) * factor\n            if offset.shape != (3,):\n                raise ValueError(\'Translation requires three metre offsets\')\n            x += offset\n            if pelvis is not None:\n                pelvis += offset\n        if mode in (\'pelvis\', \'combined\'):\n            if pelvis is None or pelvis.shape != x.shape:\n                raise ValueError(\'Synchronized pelvis_xyz [N,100,3] is required\')\n            x -= pelvis\n        elif mode in (\'translation\', \'aligned\', \'body\'):\n            x -= x[:, :1, :]\n        if mode in (\'aligned\', \'combined\'):\n            x = np.einsum(\'nti,nij->ntj\', x, gait_basis(direction))\n        if mode in (\'body\', \'combined\'):\n            if not cohort.context.get(\'units_verified\', False) or not cohort.context.get(\'body_scale_definition\'):\n                raise ValueError(\'Verify coordinate units and prespecify height or leg-length definition\')\n            height = a.get(\'body_scale_m\')\n            if height is None or height.shape != (len(x),) or np.any(height <= 0):\n                raise ValueError(\'Positive body_scale_m [N], with a fixed definition, is required\')\n            x /= height[:, None, None] * factor\n        x = x[:, :, axis:axis + 1]\n    fingerprint = sha256((ds.fingerprint + repr((mode, translation_m, rotation_deg))).encode()).hexdigest()\n    return replace(ds, x=x.astype(\'float32\'), fingerprint=fingerprint)\n\ndef write_templates(directory):\n    """Blank data contract, not fabricated external observations."""\n    root = Path(directory)\n    root.mkdir(parents=True, exist_ok=True)\n    for name, columns in {\n        \'participants.csv\': [\'participant_id\', \'global_subject_id\', \'age\', \'sex\', \'medication_state\', \'HY_stage\', \'intervention_group\'],\n        \'trials.csv\': [\'participant_id\', \'trial_id\', \'visit\', \'side\', \'include\', \'exclusion_reason\'],\n        \'cycles.csv\': [\'cycle_id\', \'participant_id\', \'trial_id\', \'visit\', \'side\', \'include\', \'exclusion_reason\'],\n    }.items():\n        if not (root / name).exists():\n            pd.DataFrame(columns=columns).to_csv(root / name, index=False)\n    context = {k: None for k in REQUIRED_CONTEXT}\n    context.update(task=None, units_verified=False, harmonization_verified=False, participant_disjointness_verified=False,\n                   canonical_axis_order=[\'forward\', \'left\', \'up\'], body_scale_definition=None)\n    if not (root / \'cohort.json\').exists():\n        write_json(root / \'cohort.json\', context)\n    return root\n\ndef harmonize_input_units(cohort, training_units):\n    """Deterministic physical-unit conversion, never fitted on external data."""\n    units = cohort.context[\'ankle_units\']\n    if units not in (\'m\', \'mm\') or training_units not in (\'m\', \'mm\'):\n        raise ValueError(\'Known internal and external length units are required\')\n    if units == training_units:\n        return cohort\n    multiplier = 1000. if units == \'m\' else .001\n    arrays = dict(cohort.arrays)\n    for key in [\'ankle_xyz\', \'pelvis_xyz\']:\n        if key in arrays:\n            arrays[key] = arrays[key] * multiplier\n    ds = replace(cohort.ds, x=cohort.ds.x * multiplier)\n    return replace(cohort, ds=ds, arrays=arrays,\n                   context={**cohort.context, \'ankle_units\': training_units})\n\ndef pack_scalar_workbooks(input_xlsx, target_xlsx, cycle_map_csv, participants_csv,\n                          trials_csv, cohort_json, output):\n    """Convert explicitly mapped old-format Excel cycles; never infer participants."""\n    import shutil\n    root = Path(output)\n    root.mkdir(parents=True, exist_ok=True)\n    ledger = read_table(cycle_map_csv, [\'cycle_id\', \'participant_id\', \'trial_id\', \'visit\', \'side\',\n                                       \'include\', \'exclusion_reason\'])\n    if {\'input_column\', \'target_column\'} - set(ledger):\n        raise ValueError(\'Explicit input_column and target_column mappings are required\')\n    accepted = ledger[bool_column(ledger.include)]\n    inputs = pd.read_excel(input_xlsx, engine=\'openpyxl\')\n    targets = pd.read_excel(target_xlsx, engine=\'openpyxl\')\n    if inputs.shape[0] != 100 or targets.shape[0] != 100:\n        raise ValueError(\'Prepare matched complete cycles at 100 normalized gait phases before packing; no padding/resampling is guessed\')\n    xs, ys = [], []\n    for row in accepted.itertuples():\n        x = inputs[row.input_column].to_numpy(dtype=\'float32\')\n        y = targets[row.target_column].to_numpy(dtype=\'float32\')\n        if not np.isfinite(x).all() or not np.isfinite(y).all():\n            raise ValueError(f\'{row.cycle_id}: missing values; document exclusion or fix the source, never zero-fill\')\n        xs.append(x[:, None]); ys.append(y[:, None])\n    if not xs:\n        raise ValueError(\'No included cycles\')\n    np.savez_compressed(root / \'waveforms.npz\', cycle_ids=accepted.cycle_id.to_numpy(str), x=np.stack(xs), y=np.stack(ys))\n    ledger.to_csv(root / \'cycles.csv\', index=False)\n    for source, filename in [(participants_csv, \'participants.csv\'), (trials_csv, \'trials.csv\'), (cohort_json, \'cohort.json\')]:\n        if Path(source).resolve() != (root / filename).resolve():\n            shutil.copy2(source, root / filename)\n    context = read_json(root / \'cohort.json\')\n    load_cohort(root, context[\'task\'], role=context[\'dataset_role\'])\n    return root\n', 'revision_models.py': '"""Subject-wise model-family selection and four explicit comparator families."""\nfrom pathlib import Path\nimport json\nimport warnings\nimport numpy as np\nimport pandas as pd\nimport joblib\nfrom sklearn.linear_model import Ridge\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.neural_network import MLPRegressor\nfrom sklearn.exceptions import ConvergenceWarning\nfrom protocol import cycle_metrics, stable_seed, scale_from_training, write_json, read_json, parameter_hash\n\nBASELINES = (\'mean_waveform\', \'ridge\', \'phase_only\', \'shallow_mlp\')\nDEEP = (\'lstm\', \'gru\', \'conv1d_lstm\')\n\ndef participant_mean(y, groups):\n    return np.mean([y[groups == g].mean(axis=0) for g in np.unique(groups)], axis=0)\n\ndef fourier_basis(timesteps, harmonics):\n    # Endpoints include 0 and 100% consistently with the supplied waveforms.\n    phase = np.linspace(0, 1, timesteps)\n    return np.column_stack([np.ones(timesteps)] +\n        [f(2 * np.pi * k * phase) for k in range(1, harmonics + 1) for f in (np.sin, np.cos)])\n\ndef baseline_grid(family, smoke=False):\n    if family == \'mean_waveform\':\n        return [{}]\n    if family == \'ridge\':\n        return [{\'alpha\': v} for v in ([1.] if smoke else [0.01, 0.1, 1., 10., 100.])]\n    if family == \'phase_only\':\n        return [{\'harmonics\': v} for v in ([3] if smoke else [1, 3, 5, 10])]\n    if family == \'shallow_mlp\':\n        return [{\'units\': u, \'alpha\': a, \'max_iter\': 30 if smoke else 1000}\n                for u in ([4] if smoke else [16, 32, 64])\n                for a in ([0.01] if smoke else [0.0001, 0.01])]\n    raise ValueError(family)\n\ndef fit_baseline(family, x, y, groups, parameters, seed):\n    obj = {\'family\': family, \'shape\': list(y.shape[1:]), \'parameters\': parameters}\n    if family == \'mean_waveform\':\n        obj[\'waveform\'] = participant_mean(y, groups)\n    elif family == \'phase_only\':\n        basis = fourier_basis(y.shape[1], parameters[\'harmonics\'])\n        # Equal participant weighting. Uses phase alone, never ankle positions.\n        coef = np.linalg.lstsq(basis, participant_mean(y, groups), rcond=None)[0]\n        obj[\'waveform\'] = basis @ coef\n    else:\n        flat = x.reshape(len(x), -1)\n        scaler = StandardScaler().fit(flat)  # Training rows only.\n        z = scaler.transform(flat)\n        target = y.reshape(len(y), -1)\n        if family == \'ridge\':\n            model = Ridge(alpha=parameters[\'alpha\'])\n            weights = np.array([1. / np.sum(groups == g) for g in groups])\n            weights *= len(weights) / weights.sum()\n            model.fit(z, target, sample_weight=weights)\n            obj[\'fit_weighting\'] = \'equal participants\'\n        elif family == \'shallow_mlp\':\n            # One hidden layer; training-loss optimizer convergence, NO random validation split.\n            model = MLPRegressor(hidden_layer_sizes=(parameters[\'units\'],), activation=\'relu\',\n                                 solver=\'lbfgs\', alpha=parameters[\'alpha\'], max_iter=parameters[\'max_iter\'],\n                                 max_fun=50000, early_stopping=False, random_state=seed)\n            with warnings.catch_warnings(record=True) as caught:\n                warnings.simplefilter(\'always\', ConvergenceWarning)\n                model.fit(z, target)\n            obj[\'convergence_warnings\'] = [str(w.message) for w in caught]\n            obj[\'fit_weighting\'] = \'equal cycles; validation score remains equal participants\'\n        else:\n            raise ValueError(family)\n        obj.update(model=model, scaler=scaler)\n    return obj\n\ndef predict_baseline(obj, x):\n    if \'waveform\' in obj:\n        return np.repeat(obj[\'waveform\'][None], len(x), axis=0)\n    z = obj[\'scaler\'].transform(x.reshape(len(x), -1))\n    return obj[\'model\'].predict(z).reshape((len(x), *obj[\'shape\']))\n\ndef backend(family, ds, cfg):\n    from models import TensorFlowBackend\n    return TensorFlowBackend(family, ds.x.shape[1], cfg[\'search_profile\'], cfg[\'epochs\'],\n                             cfg[\'patience\'], cfg[\'batch_size\'], cfg[\'verbose\'])\n\ndef cv_candidate(ds, family, parameters, cfg, directory):\n    """Only the development Dataset is passed; outer/external arrays are absent."""\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    signature = parameter_hash(parameters)\n    rows = []\n    for val in ds.subjects:\n        path = directory / f\'validation_{val}.json\'\n        if path.exists():\n            row = read_json(path)\n            if row[\'parameter_hash\'] != signature or row[\'data_fingerprint\'] != ds.fingerprint:\n                raise ValueError(\'Candidate cache changed\')\n            rows.append(row)\n            continue\n        tr, va = ds.groups != val, ds.groups == val\n        if not tr.any():\n            raise ValueError(\'At least two development participants are required\')\n        seed = stable_seed(cfg[\'seed\'], ds.task, family, val, \'inner\')\n        print(f\'    {family}: validation {val}\', flush=True)\n        if family in DEEP:\n            b = backend(family, ds, cfg)\n            (xt, xv), _ = scale_from_training(ds.x[tr], ds.x[va], enabled=cfg[\'scale_input\'])\n            fit = b.fit_inner(xt, ds.y[tr], xv, ds.y[va], parameters, seed, directory / f\'history_{val}.csv\')\n            prediction, epoch = fit[\'prediction\'], fit[\'best_epoch\']\n            convergence = []\n        else:\n            model = fit_baseline(family, ds.x[tr], ds.y[tr], ds.groups[tr], parameters, seed)\n            prediction, epoch = predict_baseline(model, ds.x[va]), None\n            convergence = model.get(\'convergence_warnings\', [])\n        metric = cycle_metrics(ds.y[va], np.asarray(prediction))\n        row = {\'validation_subject\': val, \'training_subjects\': sorted(set(ds.groups[tr])),\n               \'n_validation_cycles\': int(va.sum()), \'parameter_hash\': signature,\n               \'data_fingerprint\': ds.fingerprint, \'mae\': float(metric.mae.mean()),\n               \'rmse\': float(metric.rmse.mean()), \'best_epoch\': epoch, \'convergence_warnings\': convergence}\n        write_json(path, row)\n        rows.append(row)\n    score = float(np.mean([r[\'mae\'] for r in rows]))\n    if not np.isfinite(score):\n        raise ValueError(\'Nonfinite full subject-wise validation objective\')\n    result = {\'family\': family, \'parameters\': parameters, \'inner_subject_mae\': score, \'folds\': rows,\n              \'epochs\': max(1, int(np.ceil(np.median([r[\'best_epoch\'] for r in rows])))) if family in DEEP else None}\n    write_json(directory / \'summary.json\', result)\n    return result\n\ndef tune_family(ds, family, cfg, directory):\n    directory = Path(directory)\n    done = directory / \'selected.json\'\n    if done.exists():\n        return read_json(done)\n    directory.mkdir(parents=True, exist_ok=True)\n    if family not in DEEP:\n        candidates = [cv_candidate(ds, family, p, cfg, directory / f\'candidate_{i:03d}\')\n                      for i, p in enumerate(baseline_grid(family, cfg.get(\'smoke\', False)))]\n        selected = min(candidates, key=lambda r: (r[\'inner_subject_mae\'], parameter_hash(r[\'parameters\'])))\n    else:\n        from models import training_dependencies, search_space\n        tf, kt = training_dependencies()\n        class SubjectTuner(kt.Tuner):\n            def run_trial(self, trial, *args, **kwargs):\n                try:\n                    result = cv_candidate(ds, family, dict(trial.hyperparameters.values), cfg,\n                                          directory / \'candidates\' / trial.trial_id)\n                except (ValueError, tf.errors.ResourceExhaustedError) as exc:\n                    write_json(directory / \'candidates\' / trial.trial_id / \'failure.json\', {\'error\': str(exc)})\n                    raise kt.errors.FailedTrialError(str(exc)) from exc\n                return {\'inner_subject_mae\': result[\'inner_subject_mae\']}\n        hp = search_space(family, cfg[\'search_profile\'])\n        oracle = kt.oracles.BayesianOptimizationOracle(objective=kt.Objective(\'inner_subject_mae\', \'min\'),\n            max_trials=cfg[\'max_trials\'], num_initial_points=min(3, cfg[\'max_trials\']),\n            hyperparameters=hp, allow_new_entries=False, tune_new_entries=False,\n            seed=stable_seed(cfg[\'seed\'], ds.task, family, \'oracle\'), max_consecutive_failed_trials=3)\n        tuner = SubjectTuner(oracle=oracle, hypermodel=None, directory=str(directory), project_name=\'tuner\', overwrite=False)\n        tuner.search()\n        best = tuner.oracle.get_best_trials(1)\n        if not best or best[0].score is None or not np.isfinite(best[0].score):\n            raise ValueError(f\'No complete successful candidate for {family}\')\n        selected = read_json(directory / \'candidates\' / best[0].trial_id / \'summary.json\')\n    write_json(done, selected)\n    return selected\n\ndef fit_frozen_model(ds, selected, cfg, directory):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    descriptor = directory / \'model.json\'\n    if descriptor.exists():\n        return read_json(descriptor)\n    family = selected[\'family\']\n    seed = stable_seed(cfg[\'seed\'], ds.task, family, \'final_refit\')\n    if family in DEEP:\n        b = backend(family, ds, cfg)\n        (x,), scaler = scale_from_training(ds.x, enabled=cfg[\'scale_input\'])\n        model = b.fit_final(x, ds.y, selected[\'parameters\'], seed, selected[\'epochs\'], directory / \'history.csv\')\n        b.save(model, directory / \'model.keras\')\n        b.clear()\n        result = {\'family\': family, \'file\': \'model.keras\', \'input_scaler\': scaler}\n    else:\n        model = fit_baseline(family, ds.x, ds.y, ds.groups, selected[\'parameters\'], seed)\n        joblib.dump(model, directory / \'model.joblib\')\n        result = {\'family\': family, \'file\': \'model.joblib\',\n                  \'convergence_warnings\': model.get(\'convergence_warnings\', [])}\n    result.update(parameters=selected[\'parameters\'], epochs=selected[\'epochs\'], seed=seed,\n                  n_training_participants=len(ds.subjects), n_training_cycles=len(ds.x),\n                  training_participants=ds.subjects, data_fingerprint=ds.fingerprint)\n    write_json(descriptor, result)\n    return result\n\ndef predict_saved(directory, x):\n    directory = Path(directory)\n    info = read_json(directory / \'model.json\')\n    if info[\'family\'] in DEEP:\n        from models import training_dependencies\n        tf, _ = training_dependencies()\n        scaler = info[\'input_scaler\']\n        z = (x - np.array(scaler[\'mean\'])[None, None]) / np.array(scaler[\'std\'])[None, None]\n        model = tf.keras.models.load_model(directory / info[\'file\'], compile=False)\n        try:\n            return np.asarray(model.predict(z, verbose=0))\n        finally:\n            del model\n            tf.keras.backend.clear_session()\n    # Only load models created by this notebook, whose file hashes are frozen.\n    return np.asarray(predict_baseline(joblib.load(directory / info[\'file\']), x))\n\ndef mean_ensemble(predictions):\n    """Average predictions BEFORE errors: models are not independent participants."""\n    values = [np.asarray(p) for p in predictions]\n    if not values or any(v.shape != values[0].shape or not np.isfinite(v).all() for v in values):\n        raise ValueError(\'Ensemble members require identical finite cycle/time/component arrays\')\n    return np.mean(np.stack(values), axis=0)\n', 'evaluate_saved.py': '"""Read-only held-out evaluation. This module never trains or tunes a model."""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport json\nimport numpy as np\nimport pandas as pd\nfrom protocol import cycle_metrics\n\nFAMILIES = (\'mean_waveform\', \'ridge\', \'phase_only\', \'shallow_mlp\', \'lstm\', \'gru\', \'conv1d_lstm\')\n\ndef read_json(path):\n    return json.loads(Path(path).read_text())\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\ndef check_metadata(spec, info, selected, family, outer, ds):\n    if spec[\'config\'].get(\'smoke\', False):\n        raise ValueError(\'This evaluator requires research outputs, not smoke-test fixtures\')\n    if spec[\'task\'] != ds.task or spec[\'data_fingerprint\'] != ds.fingerprint:\n        raise ValueError(\'Internal data or preprocessing differs from the saved experiment\')\n    if sorted(spec[\'subjects\']) != ds.subjects or outer not in ds.subjects:\n        raise ValueError(\'Participant list differs from the saved experiment\')\n    if info[\'family\'] != family or selected[\'family\'] != family:\n        raise ValueError(\'Saved family mismatch\')\n    training = info.get(\'training_participants\', [])\n    expected = [s for s in ds.subjects if s != outer]\n    if outer in training or sorted(training) != expected or len(training) != len(set(training)):\n        raise ValueError(\'Test participant overlaps training, or training cohort is incomplete\')\n    if info.get(\'n_training_participants\') != len(expected):\n        raise ValueError(\'Saved training-participant count differs\')\n    if info.get(\'data_fingerprint\') != ds.subset(expected).fingerprint:\n        raise ValueError(\'Model training-data fingerprint does not match\')\n    if info.get(\'parameters\') != selected.get(\'parameters\'):\n        raise ValueError(\'Saved refit parameters differ from the frozen selection\')\n    if family in (\'lstm\', \'gru\', \'conv1d_lstm\') and info.get(\'epochs\') != selected.get(\'epochs\'):\n        raise ValueError(\'Saved refit epochs differ from the frozen selection\')\n\ndef read_prediction_cache(path, te):\n    with np.load(path, allow_pickle=False) as cache:\n        ids = cache[\'cycle_ids\'].astype(str)\n        truth = cache[\'measured\']\n        pred = cache[\'predicted\']\n    expected = te.cycles.cycle_id.to_numpy(str)\n    if not np.array_equal(ids, expected):\n        raise ValueError(\'Saved predictions have different cycle identities/order\')\n    if truth.shape != te.y.shape or pred.shape != te.y.shape:\n        raise ValueError(\'Saved prediction or target shape differs from test data\')\n    if not np.isfinite(truth).all() or not np.isfinite(pred).all():\n        raise ValueError(\'Nonfinite cached targets or predictions\')\n    if not np.allclose(truth, te.y, rtol=1e-6, atol=1e-7):\n        raise ValueError(\'Cached targets differ from the original held-out measurements\')\n    return pred\n\ndef predict_weights(directory, info, x, software):\n    """Inference only, using the input scaler saved with the fitted model."""\n    path = Path(directory)/info[\'file\']\n    if info[\'family\'] in (\'lstm\',\'gru\',\'conv1d_lstm\'):\n        for name in [\'tensorflow\',\'keras\']:\n            actual = importlib.metadata.version(name)\n            if actual != software.get(name):\n                raise ValueError(f\'{name}: inference needs the saved version {software.get(name)}, found {actual}. Cached predictions do not require TensorFlow.\')\n        import tensorflow as tf\n        scaler = info[\'input_scaler\']\n        std = np.asarray(scaler[\'std\'], float)\n        if not np.isfinite(std).all() or np.any(std <= 0):\n            raise ValueError(\'Invalid saved scaler\')\n        z = (x - np.asarray(scaler[\'mean\'])[None,None,:]) / std[None,None,:]\n        model = tf.keras.models.load_model(path, compile=False)\n        try:\n            return np.asarray(model.predict(z, verbose=0))\n        finally:\n            del model\n            tf.keras.backend.clear_session()\n    for name in [\'scikit-learn\',\'joblib\']:\n        actual = importlib.metadata.version(name)\n        if actual != software.get(name):\n            raise ValueError(f\'{name}: saved-model version is {software.get(name)}, current version is {actual}. Use cached predictions or the original package versions.\')\n    import joblib\n    # Only load trusted model files created by the user\'s own training notebook.\n    obj = joblib.load(path)\n    if obj[\'family\'] != info[\'family\']:\n        raise ValueError(\'Baseline family differs inside saved model\')\n    if \'waveform\' in obj:\n        return np.repeat(obj[\'waveform\'][None], len(x), axis=0)\n    z = obj[\'scaler\'].transform(x.reshape(len(x),-1))\n    return np.asarray(obj[\'model\'].predict(z)).reshape((len(x),*obj[\'shape\']))\n\ndef evaluate_available(root, ds, outer, output, mode=\'cached_only\'):\n    if mode not in (\'cached_only\',\'inference_if_needed\',\'fresh_inference\'):\n        raise ValueError(\'Unknown evaluation mode\')\n    root, output = Path(root), Path(output)\n    if output.resolve().is_relative_to(root.resolve()):\n        raise ValueError(\'Write test reports outside the training experiment\')\n    output.mkdir(parents=True,exist_ok=True)\n    spec = read_json(root/\'prespecification.json\')\n    te = ds.subset([outer])\n    inventory, tables, summary, predictions, hashes = [], [], [], {}, {}\n    for family in FAMILIES:\n        folder = root/\'nested\'/outer/family\n        descriptor, selection = folder/\'refit/model.json\', folder/\'selection/selected.json\'\n        if not descriptor.exists() or not selection.exists():\n            inventory.append(dict(model=family,status=\'unavailable\',reason=\'No completed frozen selection/refitted model\'))\n            continue\n        try:\n            info, chosen = read_json(descriptor), read_json(selection)\n            check_metadata(spec,info,chosen,family,outer,ds)\n            marker, cache = folder/\'complete.json\', folder/\'predictions.npz\'\n            use_cache = mode != \'fresh_inference\' and cache.exists() and marker.exists()\n            if use_cache:\n                done = read_json(marker)\n                if done.get(\'outer_subject\') != outer or done.get(\'family\') != family:\n                    raise ValueError(\'Outer-evaluation completion marker differs\')\n                pred = read_prediction_cache(cache,te)\n                kind = \'saved held-out predictions\'\n                hashes[family] = {\'predictions_sha256\':digest(cache)}\n            else:\n                if mode == \'cached_only\':\n                    inventory.append(dict(model=family,status=\'unavailable\',reason=\'Completed prediction cache absent; inference mode can use a saved refit\'))\n                    continue\n                weights = folder/\'refit\'/info[\'file\']\n                if not weights.is_file():\n                    raise ValueError(\'Refitted weights missing\')\n                pred = predict_weights(folder/\'refit\',info,te.x,spec[\'software\'])\n                kind = \'fresh inference from saved weights\'\n                hashes[family] = {\'model_sha256\':digest(weights)}\n            metrics = cycle_metrics(te.y,pred)\n            a = te.y.reshape(len(te.y),-1).astype(float)\n            b = pred.reshape(len(pred),-1).astype(float)\n            a -= a.mean(axis=1,keepdims=True)\n            b -= b.mean(axis=1,keepdims=True)\n            denominator = np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1)\n            metrics[\'waveform_r\'] = np.divide((a*b).sum(axis=1),denominator,\n                out=np.full(len(a),np.nan),where=denominator>1e-12)\n            metrics.insert(0,\'cycle_id\',te.cycles.cycle_id.to_numpy(str))\n            metrics.insert(0,\'participant_id\',outer)\n            metrics.insert(0,\'model\',family)\n            metrics.insert(0,\'task\',te.task)\n            metrics[\'prediction_source\'] = kind\n            tables.append(metrics)\n            summary.append(dict(task=te.task,participant_id=outer,model=family,n_participants=1,\n                n_cycles=len(te.y),mae=float(metrics.mae.mean()),rmse=float(metrics.rmse.mean()),\n                nrmse=float(metrics.nrmse.mean()),nrmse_percent=float(metrics.nrmse.mean()*100),\n                mean_cycle_waveform_r=float(metrics.waveform_r.mean()),\n                undefined_nrmse_cycles=int(metrics.nrmse.isna().sum()),prediction_source=kind,\n                uncertainty=\'No participant-level confidence interval from one participant\'))\n            predictions[family] = pred\n            hashes[family].update(model_metadata_sha256=digest(descriptor),selection_sha256=digest(selection))\n            np.savez_compressed(output/f\'{family}_heldout_predictions.npz\',\n                cycle_ids=te.cycles.cycle_id.to_numpy(str),measured=te.y,predicted=pred)\n            inventory.append(dict(model=family,status=\'evaluated\',reason=kind))\n        except (ValueError,KeyError,OSError,importlib.metadata.PackageNotFoundError) as error:\n            inventory.append(dict(model=family,status=\'not evaluated\',reason=str(error)))\n    inventory = pd.DataFrame(inventory)\n    inventory.to_csv(output/\'model_availability.csv\',index=False)\n    result = pd.DataFrame(summary)\n    if tables:\n        pd.concat(tables,ignore_index=True).to_csv(output/\'heldout_cycle_metrics.csv\',index=False)\n        result.to_csv(output/\'heldout_participant_metrics.csv\',index=False)\n        if \'mean_waveform\' in set(result.model):\n            reference = result.set_index(\'model\').loc[\'mean_waveform\']\n            comparisons=[]\n            for _, row in result.iterrows():\n                if row.model == \'mean_waveform\': continue\n                comparisons.append(dict(model=row.model,reference=\'mean_waveform\',n_participants=1,\n                    rmse_improvement=reference.rmse-row.rmse,\n                    nrmse_percentage_point_improvement=reference.nrmse_percent-row.nrmse_percent,\n                    interpretation=\'Positive favors model; descriptive result for one held-out participant\'))\n            pd.DataFrame(comparisons).to_csv(output/\'descriptive_baseline_comparisons.csv\',index=False)\n    (output/\'evaluation_provenance.json\').write_text(json.dumps({\n        \'experiment\':str(root),\'task\':ds.task,\'held_out_participant\':outer,\n        \'n_test_participants\':1,\'test_cycle_count\':len(te.y),\'mode\':mode,\n        \'prespecification_sha256\':digest(root/\'prespecification.json\'),\'artifacts\':hashes,\n        \'notes\':[\'No training or tuning performed\',\'No external cohort loaded\',\n                 \'No selected_deep winner declared from these test results\',\n                 \'NRMSE = cycle RMSE / measured full-cycle range; participant result = mean of defined cycle values\',\n                 \'Cycles are repeated observations within one participant; no cohort inference or significance tests\']},indent=2))\n    return inventory,result,predictions,te\n\ndef make_plots(summary, predictions, te, output):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    from matplotlib.ticker import PercentFormatter\n    from math import ceil\n    output=Path(output)\n    plt.rcParams.update({\'font.size\':10,\'axes.spines.top\':False,\'axes.spines.right\':False,\'pdf.fonttype\':42})\n    labels={\'mean_waveform\':\'Mean waveform\',\'ridge\':\'Ridge\',\'phase_only\':\'Phase only\',\n            \'shallow_mlp\':\'Shallow MLP\',\'lstm\':\'LSTM\',\'gru\':\'GRU\',\'conv1d_lstm\':\'Conv1D-LSTM\'}\n    order=[m for m in FAMILIES if m in predictions]\n    if not order: return []\n    data=summary.set_index(\'model\').loc[order]\n    colors=[\'#6C7A89\' if m not in (\'lstm\',\'gru\',\'conv1d_lstm\') else {\'lstm\':\'#0072B2\',\'gru\':\'#D58A00\',\'conv1d_lstm\':\'#009E73\'}[m] for m in order]\n    subtitle=f\'{te.task} | held-out participant {te.subjects[0]} | {len(te.y)} cycles | n = 1 participant\'\n    fig,axes=plt.subplots(1,3,figsize=(15,4.8))\n    for ax,metric,title in zip(axes,[\'mae\',\'rmse\',\'nrmse_percent\'],[\'Mean cycle MAE\',\'Mean cycle RMSE\',\'Mean cycle NRMSE (%)\']):\n        values=data[metric].to_numpy()\n        ax.barh(range(len(order)),values,color=colors)\n        ax.set_yticks(range(len(order)),[labels[m] for m in order]);ax.invert_yaxis()\n        ax.set_xlabel(\'Target units\' if metric!=\'nrmse_percent\' else \'Percent of measured cycle range\')\n        ax.set_title(title);ax.set_xlim(left=0);ax.grid(axis=\'x\',alpha=.15)\n    fig.suptitle(\'Saved-model held-out errors\\n\'+subtitle,fontsize=14)\n    fig.text(.02,.015,\'Descriptive participant-level results. No cohort confidence intervals. Physical target units require source verification.\',fontsize=9)\n    fig.tight_layout(rect=(0,.07,1,.89))\n    fig.savefig(output/\'Heldout_Model_Errors.png\',dpi=180)\n    fig.savefig(output/\'Heldout_Model_Errors.pdf\');plt.close(fig)\n    phase=np.linspace(0,100,te.y.shape[1]);truth=te.y[:,:,0]\n    fig,axes=plt.subplots(ceil(len(order)/3),3,figsize=(15,3.5*ceil(len(order)/3)),squeeze=False,sharex=True,sharey=True)\n    for ax,family,color in zip(axes.flat,order,colors):\n        low,high=np.quantile(truth,[.25,.75],axis=0)\n        ax.fill_between(phase,low,high,color=\'#ADB5BD\',alpha=.25,label=\'Measured cycle IQR\')\n        ax.plot(phase,truth.mean(axis=0),color=\'black\',lw=1.8,label=\'Measured mean\')\n        ax.plot(phase,predictions[family][:,:,0].mean(axis=0),color=color,lw=1.8,label=\'Predicted mean\')\n        ax.set_title(labels[family]);ax.set_xlabel(\'Normalized gait phase (%)\');ax.set_ylabel(\'GRF (target units)\')\n        ax.grid(alpha=.15)\n    for ax in list(axes.flat)[len(order):]:ax.set_visible(False)\n    handles,legend=axes.flat[0].get_legend_handles_labels()\n    fig.legend(handles,legend,loc=\'lower center\',ncol=3,frameon=False)\n    fig.suptitle(\'Measured and predicted waveforms\\n\'+subtitle,fontsize=14)\n    fig.tight_layout(rect=(0,.07,1,.90))\n    fig.savefig(output/\'Heldout_Waveforms.png\',dpi=180)\n    fig.savefig(output/\'Heldout_Waveforms.pdf\');plt.close(fig)\n    k=min(3,len(truth));fig,axes=plt.subplots(k,1,figsize=(11,3*k),squeeze=False,sharex=True)\n    for i,ax in enumerate(axes[:,0]):\n        ax.plot(phase,truth[i],color=\'black\',lw=2.2,label=\'Measured\')\n        for j,(family,color) in enumerate(zip(order,colors)):\n            ax.plot(phase,predictions[family][i,:,0],color=color,ls=[\'--\',\':\',\'-.\',\'--\',\'-\',\'-\',\'-\'][FAMILIES.index(family)],lw=1.4,label=labels[family])\n        ax.set_title(str(te.cycles.cycle_id.iloc[i]));ax.set_ylabel(\'GRF (target units)\');ax.grid(alpha=.15)\n    axes[-1,0].set_xlabel(\'Normalized gait phase (%)\')\n    fig.legend(*axes[0,0].get_legend_handles_labels(),loc=\'lower center\',ncol=4,frameon=False)\n    fig.suptitle(\'First available cycles in source order; no selection by prediction quality\\n\'+subtitle,fontsize=12)\n    fig.tight_layout(rect=(0,.10,1,.91))\n    fig.savefig(output/\'Heldout_Example_Cycles.png\',dpi=180);plt.close(fig)\n    return [output/name for name in [\'Heldout_Model_Errors.png\',\'Heldout_Waveforms.png\',\'Heldout_Example_Cycles.png\']]\n', 'seed_stability.py': '"""Frozen-setting repeated-seed LOSO sensitivity study. No hyperparameter search."""\nfrom pathlib import Path\nimport hashlib,json,math,time,traceback,importlib.metadata,platform\nimport numpy as np,pandas as pd\nfrom data import TASK_ORDER\nfrom revision_data import legacy_cohort,coordinate_dataset\nfrom protocol import cycle_metrics,scale_from_training\nfrom evaluate_saved import check_metadata\n\nFAMILIES=(\'lstm\',\'gru\',\'conv1d_lstm\')\ndef digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef hashobj(o):return hashlib.sha256(json.dumps(o,sort_keys=True,allow_nan=False).encode()).hexdigest()\ndef read(p):return json.loads(Path(p).read_text())\ndef write(p,o):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+\'.tmp\');tmp.write_text(json.dumps(o,indent=2,allow_nan=False));tmp.replace(p)\ndef integer_seed(master,*labels):return int(hashlib.sha256((str(master)+\'|\'+\'|\'.join(map(str,labels))).encode()).hexdigest()[:8],16)%(2**31-1)\ndef versions():\n d={\'python\':platform.python_version()}\n for name in [\'tensorflow\',\'keras\',\'keras-tuner\',\'numpy\',\'pandas\',\'openpyxl\']:\n  try:d[name]=importlib.metadata.version(name)\n  except importlib.metadata.PackageNotFoundError:d[name]=None\n return d\n\ndef inventory(data_zip,source_root,tasks,families,outer,code_root):\n cohorts={};jobs=[];rows=[]\n for task in tasks:\n  cohort=legacy_cohort(data_zip,task);ds=coordinate_dataset(cohort,\'absolute\');cohorts[task]=ds\n  root=Path(source_root)/task/\'absolute\';sp=root/\'prespecification.json\'\n  if not sp.is_file():\n   rows.append(dict(task=task,outer_subject=\'all\',family=\'all\',status=\'unavailable\',reason=\'Source prespecification missing\'));continue\n  spec=read(sp)\n  if spec[\'task\']!=task or spec[\'data_fingerprint\']!=ds.fingerprint or sorted(spec[\'subjects\'])!=ds.subjects or spec[\'config\'].get(\'smoke\'):\n   raise ValueError(f\'{task}: source data or research profile mismatch\')\n  for file in [\'data.py\',\'protocol.py\',\'models.py\',\'revision_data.py\',\'revision_models.py\']:\n   if digest(Path(code_root)/file)!=spec[\'sources\'].get(file):raise ValueError(f\'{task}: source-code mismatch for {file}\')\n  for sub in ds.subjects:\n   if outer!=\'all\' and sub not in outer:continue\n   for family in families:\n    folder=root/\'nested\'/sub/family;sel=folder/\'selection/selected.json\';inf=folder/\'refit/model.json\';done=folder/\'complete.json\'\n    if not all(p.is_file() for p in [sel,inf,done]):\n     rows.append(dict(task=task,outer_subject=sub,family=family,status=\'unavailable\',reason=\'Completed original selection/refit/outer marker missing\'));continue\n    selected,info=read(sel),read(inf);check_metadata(spec,info,selected,family,sub,ds)\n    expected=[s for s in ds.subjects if s!=sub];folds=selected.get(\'folds\',[])\n    if sorted(r[\'validation_subject\'] for r in folds)!=expected:raise ValueError(\'Incomplete inner subject-wise selection\')\n    train=ds.subset(expected)\n    for r in folds:\n     if sorted(r[\'training_subjects\'])!=[s for s in expected if s!=r[\'validation_subject\']] or r[\'data_fingerprint\']!=train.fingerprint or not np.isfinite(r[\'mae\']):raise ValueError(\'Invalid inner fold data/separation\')\n    epoch=max(1,int(np.ceil(np.median([r[\'best_epoch\'] for r in folds]))))\n    if epoch!=selected[\'epochs\'] or epoch!=info[\'epochs\']:raise ValueError(\'Epoch budget is not the fixed inner-derived budget\')\n    job=dict(task=task,outer_subject=sub,family=family,parameters=selected[\'parameters\'],epochs=epoch,batch_size=int(spec[\'config\'][\'batch_size\']),scale_input=bool(spec[\'config\'][\'scale_input\']),search_profile=spec[\'config\'][\'search_profile\'],source_selection_sha256=digest(sel),source_descriptor_sha256=digest(inf),source_spec_sha256=digest(sp),source_software=spec[\'software\'],data_fingerprint=ds.fingerprint)\n    rows.append(dict(task=task,outer_subject=sub,family=family,status=\'available\',reason=\'\',epochs=epoch));jobs.append(job)\n return cohorts,jobs,pd.DataFrame(rows)\n\ndef bind_plan(output,settings,jobs,code_root):\n out=Path(output);out.mkdir(parents=True,exist_ok=True)\n signature={\'settings\':settings,\'jobs\':jobs,\'software\':versions(),\'code_sha256\':{p.name:digest(p) for p in Path(code_root).glob(\'*.py\')}}\n p=out/\'frozen_seed_plan.json\'\n if p.exists() and read(p)!=signature:raise ValueError(\'Data, settings, selected configurations, software or code changed. Use a NEW RUN_ID; do not mix results.\')\n if not p.exists():write(p,signature)\n return hashobj(signature)\n\ndef train_job(ds,job,master_seed,output,plan_hash,verbose=0):\n import tensorflow as tf\n from models import build_model\n sub=job[\'outer_subject\'];fam=job[\'family\'];folder=Path(output)/\'runs\'/ds.task/sub/fam/f\'seed_{master_seed}\';folder.mkdir(parents=True,exist_ok=True)\n completed=folder/\'result.json\';cache=folder/\'predictions.npz\'\n if completed.exists():\n  result=read(completed)\n  if result[\'plan_hash\']!=plan_hash or result[\'master_seed\']!=master_seed or not cache.exists() or result[\'predictions_sha256\']!=digest(cache):raise ValueError(\'Completed result/cache mismatch\')\n  return result\n actual_seed=integer_seed(master_seed,ds.task,sub,fam,\'initialization\');tr=np.flatnonzero(ds.groups!=sub);te=np.flatnonzero(ds.groups==sub)\n assert len(tr)>0 and len(te)>0 and not set(ds.groups[tr])&set(ds.groups[te])\n (xtr,xte),scaler=scale_from_training(ds.x[tr],ds.x[te],enabled=job[\'scale_input\']);ytr=ds.y[tr]\n steps=math.ceil(len(tr)/job[\'batch_size\']);checkpoint=folder/\'checkpoint.keras\';start=time.monotonic()\n tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(actual_seed)\n if checkpoint.exists():\n  model=tf.keras.models.load_model(checkpoint)\n  iters=int(model.optimizer.iterations.numpy())\n  if iters%steps:raise ValueError(\'Checkpoint does not represent a complete epoch\')\n  epoch=iters//steps\n else:model=build_model(fam,ds.x.shape[1],job[\'parameters\'],job[\'search_profile\']);epoch=0\n if epoch>job[\'epochs\']:raise ValueError(\'Checkpoint exceeded frozen epoch budget\')\n print(f\'{ds.task}/{sub}/{fam}/seed {master_seed}: {epoch}/{job["epochs"]} epochs\',flush=True)\n try:\n  while epoch<job[\'epochs\']:\n   e_seed=integer_seed(master_seed,ds.task,sub,fam,\'epoch\',epoch+1);tf.keras.utils.set_random_seed(e_seed)\n   order=np.random.default_rng(e_seed).permutation(len(tr));before=int(model.optimizer.iterations.numpy())\n   h=model.fit(xtr[order],ytr[order],batch_size=job[\'batch_size\'],epochs=1,shuffle=False,verbose=verbose,callbacks=[tf.keras.callbacks.TerminateOnNaN()])\n   loss=float(h.history[\'loss\'][-1])\n   if not np.isfinite(loss) or int(model.optimizer.iterations.numpy())!=before+steps:raise ValueError(\'Nonfinite or incomplete training epoch; last good checkpoint retained\')\n   # Save weights and optimizer state every epoch. Optimizer iterations are authoritative.\n   tmp=folder/\'checkpoint.tmp.keras\';model.save(tmp);tmp.replace(checkpoint);epoch+=1\n   write(folder/\'training_progress.json\',dict(master_seed=master_seed,actual_seed=actual_seed,completed_epochs=epoch,total_epochs=job[\'epochs\'],last_loss=loss,plan_hash=plan_hash))\n   with (folder/\'history.csv\').open(\'a\') as stream:\n    if stream.tell()==0:stream.write(\'epoch,loss,epoch_seed\\n\')\n    stream.write(f\'{epoch},{loss},{e_seed}\\n\')\n   if epoch==1 or epoch%10==0 or epoch==job[\'epochs\']:print(f\'  epoch {epoch}/{job["epochs"]}; loss {loss:.6g}\',flush=True)\n  pred=np.asarray(model.predict(xte,batch_size=job[\'batch_size\'],verbose=0));metric=cycle_metrics(ds.y[te],pred)\n  if not np.isfinite(metric.rmse).all():raise ValueError(\'Nonfinite held-out metric\')\n  ids=ds.cycles.iloc[te].cycle_id.to_numpy(str);np.savez_compressed(cache,measured=ds.y[te],predicted=pred,cycle_ids=ids,subject_ids=ds.groups[te].astype(str),gait_phase=np.linspace(0,100,ds.y.shape[1]))\n  metric.insert(0,\'cycle_id\',ids);metric.to_csv(folder/\'cycle_metrics.csv\',index=False)\n  result={**job,\'master_seed\':master_seed,\'actual_seed\':actual_seed,\'participant\':sub,\'n_cycles\':len(te),\'n_training_cycles\':len(tr),\'n_training_participants\':len(set(ds.groups[tr])),\'rmse\':float(metric.rmse.mean()),\'nrmse\':float(metric.nrmse.mean()) if metric.nrmse.notna().any() else None,\'mae\':float(metric.mae.mean()),\'undefined_nrmse_cycles\':int((~metric.nrmse_defined).sum()),\'input_scaler\':scaler,\'elapsed_this_session_seconds\':time.monotonic()-start,\'plan_hash\':plan_hash,\'predictions_sha256\':digest(cache)}\n  write(completed,result);return result\n finally:\n  del model;tf.keras.backend.clear_session()\n\ndef summarize(output,jobs,seeds,cohorts):\n out=Path(output);rows=[];coverage=[]\n for job in jobs:\n  complete=[]\n  for seed in seeds:\n   p=out/\'runs\'/job[\'task\']/job[\'outer_subject\']/job[\'family\']/f\'seed_{seed}\'/\'result.json\'\n   if p.exists():\n    r=read(p);complete.append(seed);rows.append({k:r[k] for k in [\'task\',\'family\',\'participant\',\'master_seed\',\'actual_seed\',\'epochs\',\'n_cycles\',\'rmse\',\'nrmse\',\'mae\',\'undefined_nrmse_cycles\']})\n  coverage.append(dict(task=job[\'task\'],family=job[\'family\'],participant=job[\'outer_subject\'],completed_seeds=len(complete),expected_seeds=len(seeds),all_seeds_complete=set(complete)==set(seeds)))\n cv=pd.DataFrame(coverage);cv.to_csv(out/\'coverage.csv\',index=False)\n if not rows:return None\n df=pd.DataFrame(rows);df.to_csv(out/\'participant_seed_metrics.csv\',index=False)\n # Balanced reporting: only participants with every requested seed enter summaries.\n good=cv[cv.all_seeds_complete];balanced=df.merge(good[[\'task\',\'family\',\'participant\']],on=[\'task\',\'family\',\'participant\'])\n if balanced.empty:return None\n participant=[];seedrows=[];stability=[]\n for (task,family,sub),g in balanced.groupby([\'task\',\'family\',\'participant\']):\n  for metric in [\'rmse\',\'nrmse\',\'mae\']:\n   a=g[metric].dropna().to_numpy();participant.append(dict(task=task,family=family,participant=sub,metric=metric,n_seeds=len(a),mean=a.mean() if len(a) else np.nan,sd=a.std(ddof=1) if len(a)>1 else np.nan,minimum=a.min() if len(a) else np.nan,maximum=a.max() if len(a) else np.nan))\n pd.DataFrame(participant).to_csv(out/\'participant_seed_variation.csv\',index=False)\n for (task,family),g in balanced.groupby([\'task\',\'family\']):\n  folds=g.participant.nunique();total=len(cohorts[task].subjects)\n  for seed,gg in g.groupby(\'master_seed\'):\n   seedrows.append(dict(task=task,family=family,master_seed=seed,n_participants=folds,expected_cohort_participants=total,full_cohort_coverage=folds==total,rmse=gg.rmse.mean(),nrmse=gg.nrmse.mean(),mae=gg.mae.mean()))\n sd=pd.DataFrame(seedrows);sd.to_csv(out/\'cohort_metrics_per_seed.csv\',index=False)\n for (task,family),g in sd.groupby([\'task\',\'family\']):\n  for metric in [\'rmse\',\'nrmse\',\'mae\']:\n   a=g[metric].dropna().to_numpy();stability.append(dict(task=task,family=family,metric=metric,n_seeds=len(a),n_participants=int(g.n_participants.iloc[0]),expected_cohort_participants=int(g.expected_cohort_participants.iloc[0]),scope=\'full cohort\' if g.full_cohort_coverage.all() else \'limited available folds\',mean=a.mean() if len(a) else np.nan,sd=a.std(ddof=1) if len(a)>1 else np.nan,minimum=a.min() if len(a) else np.nan,maximum=a.max() if len(a) else np.nan))\n pd.DataFrame(stability).to_csv(out/\'seed_stability_summary.csv\',index=False);return pd.DataFrame(stability)\n'}
for name,text in BUNDLED_FILES.items():
    (CODE/name).write_text(text)
if str(CODE) not in sys.path:sys.path.insert(0,str(CODE))
os.environ['TF_DETERMINISTIC_OPS']='1'
import tensorflow as tf
from models import configure_tensorflow
hardware=configure_tensorflow(SEEDS[0])
print(hardware)
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then rerun. CPU training would be substantially slower.')


In [ ]:
from seed_stability import inventory,bind_plan,versions,write
from data import TASK_ORDER
assert TASKS and not set(TASKS)-set(TASK_ORDER) and len(set(TASKS))==len(TASKS)
assert FAMILIES and not set(FAMILIES)-{'lstm','gru','conv1d_lstm'} and len(set(FAMILIES))==len(FAMILIES)
assert OUTER_PARTICIPANTS=='all' or (isinstance(OUTER_PARTICIPANTS,list) and OUTER_PARTICIPANTS)
cohorts,jobs,available=inventory(DATA_ZIP,SOURCE_ROOT,TASKS,FAMILIES,OUTER_PARTICIPANTS,CODE)
OUTPUT.mkdir(parents=True,exist_ok=True)
available.to_csv(OUTPUT/'selection_inventory.csv',index=False)
display(available[available.status=='available'])
print(available.groupby(['task','status']).size().to_string())
if not jobs:raise RuntimeError('No compatible completed selections are available; training cannot start. Complete original nested selection first.')
settings={'seeds':SEEDS,'tasks':TASKS,'families':FAMILIES,'outer':OUTER_PARTICIPANTS,
          'seed_rule':'SHA256 master/task/outer/family labels; independent initialization and epoch permutation',
          'checkpoint_rule':'every complete epoch; optimizer iterations determine resume epoch',
          'reporting_rule':'all requested seeds; balanced participant set; equal participant weighting',
          'scope':'frozen-setting internal training sensitivity; no hyperparameter reselection'}
plan_hash=bind_plan(OUTPUT,settings,jobs,CODE)
write(OUTPUT/'hardware.json',{'tensorflow':hardware,'devices':[d.name for d in tf.config.list_physical_devices()]})
print('Configurations:',len(jobs),'planned model fits:',len(jobs)*len(SEEDS))
print('Total training epochs across all seeds:',sum(j['epochs'] for j in jobs)*len(SEEDS))
print('One complete epoch per fit is retained on Drive; do not change RUN_ID settings after the first run.')


## Train all planned seeds

This cell trains from fresh initialization and never uses the outer participant for tuning, early stopping or training duration. It saves the previous valid epoch if a nonfinite epoch occurs. Failed runs are logged and other configurations continue; they remain visible in the report. Run this cell again after restarting to resume incomplete runs. Epoch ordering is generated independently from fixed seed labels, so resume uses the same next permutation. Identical software and hardware are preferred; cross-device bitwise identity is not guaranteed.

Large Keras checkpoints saved to Drive each epoch add I/O overhead. The first fit supplies a useful runtime estimate. The notebook does not automatically repeat the hyperparameter search for missing models.

In [ ]:
from seed_stability import train_job,summarize,write
import time, traceback
start=time.monotonic()
failures=[]
for job in jobs:
    for master_seed in SEEDS:
        try:
            result=train_job(cohorts[job['task']],job,master_seed,OUTPUT,plan_hash,verbose=TRAIN_VERBOSE)
            print('Complete:',job['task'],job['outer_subject'],job['family'],master_seed,
                  'RMSE=',round(result['rmse'],6),flush=True)
        except Exception as exc:
            failure={'task':job['task'],'participant':job['outer_subject'],'family':job['family'],
                     'master_seed':master_seed,'error':str(exc),'traceback':traceback.format_exc()}
            failures.append(failure)
            write(OUTPUT/'failures_this_session.json',failures)
            print('FAILED (retained in report):',failure['task'],failure['participant'],failure['family'],master_seed,str(exc),flush=True)
        summarize(OUTPUT,jobs,SEEDS,cohorts)
write(OUTPUT/'failures_this_session.json',failures)
print('Session minutes:',round((time.monotonic()-start)/60,1),'failed/incomplete fits:',len(failures))


## Report training variability

The mean, sample SD and range across seeds refer to cohort performance calculated separately for each seed, first averaging cycle errors within each participant and then averaging participants equally. SD across seeds is **not** a participant-bootstrap confidence interval. NRMSE uses each cycle's measured force range and is stored as a fraction; plots convert it to percent.

Only folds complete for every requested seed enter the balanced summaries; coverage reports list all missing seed runs. If any scheduled fold is excluded, reports concern the balanced completed subset, not the full scheduled study. Full-cohort coverage requires every participant for the corresponding task/model. Missing architectures or tasks are not assigned zero performance. Architecture comparisons require identical participant coverage; the seed-stability summaries are calculated separately within each architecture. These runs are not used to pick a best architecture, seed or epoch budget.

In [ ]:
import pandas as pd, numpy as np
from seed_stability import summarize
summary=summarize(OUTPUT,jobs,SEEDS,cohorts)
coverage=pd.read_csv(OUTPUT/'coverage.csv')
display(coverage)
if summary is None:
    print('No fold has completed every requested seed. Resume training; no balanced stability claim is available yet.')
else:
    display(summary)
    print('Incomplete scheduled folds:',int((~coverage.all_seeds_complete).sum()))
    print('Reported scope:',summary[['task','family','n_participants','expected_cohort_participants','scope']].drop_duplicates().to_string(index=False))


In [ ]:
# Summary plot: individual seed values and mean ± sample SD across seeds.
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
per_seed_path=OUTPUT/'cohort_metrics_per_seed.csv'
if summary is not None and per_seed_path.is_file():
    per_seed=pd.read_csv(per_seed_path)
    tasks=per_seed.task.unique().tolist()
    fig,axs=plt.subplots(len(tasks),2,figsize=(11,4*len(tasks)),squeeze=False,layout='constrained')
    for i,task in enumerate(tasks):
        g=per_seed[per_seed.task==task]
        for j,metric in enumerate(['rmse','nrmse']):
            ax=axs[i,j];families=g.family.unique().tolist()
            for k,family in enumerate(families):
                vals=g[g.family==family][metric].dropna().to_numpy()*(100 if metric=='nrmse' else 1)
                ax.scatter(k+np.linspace(-.1,.1,len(vals)),vals,color='#176b8b',s=28)
                ax.errorbar(k,vals.mean(),yerr=vals.std(ddof=1),fmt='D',color='#bb632b',capsize=5)
            n=g.n_participants.iloc[0];full=g.full_cohort_coverage.all()
            ax.set_xticks(range(len(families)),[f'{f}\n(n={int(g[g.family==f].n_participants.iloc[0])})' for f in families])
            ax.set_title(f'{task}: {"full cohort" if full else "available folds"}')
            ax.set_ylabel('NRMSE (%)' if metric=='nrmse' else 'RMSE (original target units)')
            ax.spines[['top','right']].set_visible(False)
    fig.suptitle('Repeated-seed stability: dots = seeds; diamonds/whiskers = mean ± SD')
    fig.savefig(OUTPUT/'Seed_Stability_Summary.png',dpi=200)
    fig.savefig(OUTPUT/'Seed_Stability_Summary.pdf')
    plt.show()


In [ ]:
# Export reports and audit metadata; full checkpoints remain in Drive.
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
report_zip=OUTPUT/'Seed_Stability_Reports.zip'
with ZipFile(report_zip,'w',ZIP_DEFLATED) as z:
    for pattern in ['*.csv','*.json','*.png','*.pdf']:
        for p in OUTPUT.glob(pattern):z.write(p,p.name)
    for p in (OUTPUT/'runs').rglob('result.json'):z.write(p,str(p.relative_to(OUTPUT)))
    for p in (OUTPUT/'runs').rglob('training_progress.json'):z.write(p,str(p.relative_to(OUTPUT)))
    for p in (OUTPUT/'runs').rglob('history.csv'):z.write(p,str(p.relative_to(OUTPUT)))
    for p in CODE.glob('*.py'):z.write(p,'code/'+p.name)
print('Reports:',report_zip)
print('Optimizer checkpoints and held-out predictions remain under:',OUTPUT/'runs')
files.download(str(report_zip))


## How to use the results in the reviewer response

Report the actual completed seeds and participant coverage. For each architecture/task, quote the across-seed mean ± SD and range of participant-averaged RMSE and NRMSE. Do not describe a SUB01-only run as full-cohort stability. Keep single-participant stability separate from cohort inference. Failure rates and incomplete configurations must be disclosed.

Suggested wording after completion: “We repeated training using five fixed random seeds while retaining the internally selected hyperparameters, epoch budgets and participant splits. We report performance for every seed and the mean, SD and range across seeds. This sensitivity analysis quantifies stochastic training variation conditional on the fixed selected configurations.” Add the actual evaluated folds and scope, and do not claim the original 3,600-model search was repeated.

This notebook has undergone local syntax, data-loader and reporting checks. TensorFlow training must be executed and verified on Colab; it was not run in the preparation environment.